# Day3_02A_Agent_As_Tool_vs_Handoff

## OpenAI Agents SDK - Advanced Agent Pattern

**Duration:** 20 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Theme:** When should Agents collaborate, and how?

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand the difference between **Handoff** and **Agent-as-Tool**.
- Explain when to route a user to another Agent.
- Explain when to use specialist Agents as internal helpers.
- Build a simple DSU-style multi-agent teaching assistant.
- Relate the same pattern to enterprise AI systems.

---

## Previous Notebook Recap

So far, we have learned:

- How to create a basic Agent.
- How to use structured output.
- How to add Function Tools.
- How to add guardrails and tracing.
- How to build simple multi-agent handoffs.

Now we will go one level deeper.

We will answer an important question:

> When should an Agent **handoff** the conversation, and when should it **use another Agent as a tool**?


# 1. Real-World Story: DSU Faculty Assistant

Imagine a faculty member asks an AI assistant:

> "Prepare tomorrow's class on Agentic AI."

This task may require many activities:

- Explain the topic.
- Create a short quiz.
- Suggest a coding demo.
- Recommend references.

Should one Agent do everything?

It can, but it may not be ideal.

In real institutions, work is divided among specialists:

- Subject Expert
- Quiz Creator
- Lab Instructor
- Academic Advisor

Multi-agent design follows the same idea.


# 2. Two Collaboration Patterns

There are two common ways Agents collaborate.

## Pattern 1: Handoff

The first Agent sends the user to another specialist Agent.

Example:

```text
Student
   ↓
Reception Agent
   ↓
Academic Advisor / Fee Advisor / Placement Advisor
   ↓
Specialist responds directly
```

The original Agent does not complete the task.  
It routes the user to the right specialist.

---

## Pattern 2: Agent-as-Tool

The main Agent stays in control and internally calls specialist Agents.

Example:

```text
Faculty
   ↓
Teaching Orchestrator
   ├── Explainer Agent
   ├── Quiz Agent
   └── Code Demo Agent
   ↓
Combined teaching material
```

The user still interacts with the main Agent.  
The specialist Agents support the main Agent internally.


# 3. Simple Comparison

| Situation | Best Pattern | Why |
|---|---|---|
| Student asks about semester fees | Handoff | Fee Advisor should answer directly |
| Student asks about placement process | Handoff | Placement Advisor is the specialist |
| Faculty asks for complete teaching material | Agent-as-Tool | Multiple specialists contribute |
| Faculty asks for quiz + explanation + demo | Agent-as-Tool | Orchestrator combines outputs |
| Customer asks for refund status | Handoff | Customer Support Agent can handle |
| Manager asks for full market research report | Agent-as-Tool | Multiple research Agents contribute |

---

## Simple Mental Model

**Handoff** = Reception desk routing to the right department.  
**Agent-as-Tool** = Project manager collecting inputs from specialists.


# Pause & Reflect

Ask the class:

> Suppose a student asks: "What are the semester fees?"

Should the Teaching Agent answer?

Probably not.

It should handoff to the Fee Advisor.

Now ask:

> Suppose a professor asks: "Prepare complete teaching material for tomorrow."

Should the Agent handoff?

No.

It should coordinate multiple specialist Agents and return a combined response.


# 4. Setup

We will use the OpenAI Agents SDK.

Important:

In VS Code/Jupyter notebooks, use:

```python
await Runner.run(...)
```

instead of:

```python
Runner.run_sync(...)
```

because notebooks already run an event loop.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

from agents import Agent, Runner

# 5. Load Environment Variables

We will use the same root `.env` loading pattern used across all FDP notebooks.


In [ ]:
current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

# 6. Create Specialist Agents

For this notebook, we will create three specialist Agents:

1. **Explainer Agent**  
   Explains concepts simply.

2. **Quiz Agent**  
   Creates classroom questions.

3. **Code Demo Agent**  
   Suggests simple Python demo ideas.

These are like academic specialists helping a faculty member prepare a class.


In [ ]:
explainer_agent = Agent(
    name="Concept Explainer Agent",
    instructions="""
    You are a concept explanation specialist for engineering faculty.
    Explain technical topics in simple classroom-friendly language.
    Use short examples suitable for undergraduate engineering students.
    """,
)

quiz_agent = Agent(
    name="Quiz Creator Agent",
    instructions="""
    You are a quiz creation specialist for engineering faculty.
    Create short quizzes with answers.
    Keep questions beginner-friendly and useful for classroom discussion.
    """,
)

code_agent = Agent(
    name="Code Demo Agent",
    instructions="""
    You are a coding demo specialist for engineering faculty.
    Suggest simple Python notebook demo ideas.
    Keep demos short, practical, and suitable for live classroom teaching.
    """,
)

# 7. First Run: Specialist Agents Individually

Before combining Agents, let us run each specialist separately.

This helps us understand what each Agent contributes.


In [ ]:
result = await Runner.run(
    explainer_agent,
    "Explain Agent-as-Tool in Agentic AI in 3 simple points."
)

print(result.final_output)

In [ ]:
result = await Runner.run(
    quiz_agent,
    "Create 3 beginner MCQs on Agent-as-Tool vs Handoff."
)

print(result.final_output)

In [ ]:
result = await Runner.run(
    code_agent,
    "Suggest one simple Python demo idea for Agent-as-Tool."
)

print(result.final_output)

## What did we just learn?

Each specialist Agent is focused on one type of work.

This has three benefits:

- Better clarity
- Easier maintenance
- Easier debugging

But running each Agent manually is not enough.

Next, we need an orchestrator.


# 8. Agent-as-Tool Pattern

In the Agent-as-Tool pattern, the main Agent uses other Agents as internal helpers.

For teaching clarity, we will implement the orchestration using Python.

The flow is:

```text
Faculty Request
   ↓
Teaching Orchestrator
   ├── Ask Explainer Agent
   ├── Ask Quiz Agent
   └── Ask Code Demo Agent
   ↓
Combined response
```

The faculty member receives one combined answer.


In [ ]:
async def teaching_orchestrator(topic: str):
    """
    Agent-as-Tool style orchestration.

    The main function coordinates multiple specialist Agents
    and combines their outputs into one teaching pack.
    """

    explanation = await Runner.run(
        explainer_agent,
        f"Explain {topic} in simple terms for engineering faculty."
    )

    quiz = await Runner.run(
        quiz_agent,
        f"Create 3 beginner MCQs with answers on {topic}."
    )

    demo = await Runner.run(
        code_agent,
        f"Suggest one short live notebook demo idea for {topic}."
    )

    combined_output = f"""
# Teaching Pack: {topic}

## 1. Concept Explanation

{explanation.final_output}

---

## 2. Classroom Quiz

{quiz.final_output}

---

## 3. Live Demo Idea

{demo.final_output}
"""

    return combined_output

# 9. Run the Agent-as-Tool Demo

Now we ask for a complete teaching pack.

The orchestrator internally uses multiple specialist Agents and combines the result.


In [ ]:
teaching_pack = await teaching_orchestrator("Handoff vs Agent-as-Tool in Agentic AI")

print(teaching_pack)

## What did we just learn?

In Agent-as-Tool:

- The user does not directly talk to every specialist.
- The main orchestrator remains in control.
- Specialist Agents contribute internally.
- The final output is combined and returned to the user.

This pattern is useful when the final answer needs inputs from multiple experts.


# 10. Handoff Pattern

Now let us understand Handoff.

In Handoff, the first Agent does not solve everything.

It routes the user to the correct specialist.

Example:

```text
Student
   ↓
Reception Agent
   ├── Academic Advisor
   ├── Fee Advisor
   └── Placement Advisor
   ↓
One specialist responds
```

This is useful when the user's request belongs clearly to one department.


# 11. Create Handoff Specialist Agents

We will create three DSU-style support Agents:

- Academic Advisor
- Fee Advisor
- Placement Advisor


In [ ]:
academic_advisor = Agent(
    name="Academic Advisor",
    instructions="""
    You answer academic questions such as syllabus, class schedule,
    course outcomes, assignments, and learning support.
    """,
)

fee_advisor = Agent(
    name="Fee Advisor",
    instructions="""
    You answer questions related to college fees, payment dates,
    fee structure, and payment process.
    Do not invent exact amounts. Ask the student to confirm with the official office if needed.
    """,
)

placement_advisor = Agent(
    name="Placement Advisor",
    instructions="""
    You answer questions related to campus placements, training,
    resumes, interviews, and company preparation.
    """,
)

# 12. Simple Handoff Router

For classroom teaching, we will implement a simple router.

The router reads the user request and sends it to the right specialist Agent.

In production systems, this routing can be done using:

- LLM-based routing
- Classification models
- Business rules
- User role and access permissions
- Workflow engines


In [ ]:
async def handoff_router(user_request: str):
    """
    Simple DSU-style handoff router.

    This simulates routing a student query to the right specialist Agent.
    """

    request = user_request.lower()

    if "fee" in request or "payment" in request or "semester amount" in request:
        chosen_agent = fee_advisor

    elif "placement" in request or "company" in request or "interview" in request:
        chosen_agent = placement_advisor

    elif "syllabus" in request or "assignment" in request or "course" in request or "class" in request:
        chosen_agent = academic_advisor

    else:
        return "Reception Agent: I am not sure which department should handle this. Please clarify your request."

    result = await Runner.run(
        chosen_agent,
        user_request
    )

    return f"Routed to: {chosen_agent.name}\n\n{result.final_output}"

# 13. Run Handoff Examples

In these examples, only one specialist Agent answers the user's query.


In [ ]:
answer = await handoff_router(
    "When is the next assignment submission for the Agentic AI course?"
)

print(answer)

In [ ]:
answer = await handoff_router(
    "What is the process for semester fee payment?"
)

print(answer)

In [ ]:
answer = await handoff_router(
    "How should I prepare for campus placement interviews?"
)

print(answer)

## What did we just learn?

In Handoff:

- The first Agent acts like a reception desk.
- The request is routed to one specialist.
- The specialist responds.
- This is useful when the user belongs to one clear path.

Handoff is not mainly for combining many outputs.

It is mainly for routing.


# 14. Handoff vs Agent-as-Tool: Practical Decision Guide

Use this simple question:

> Does the user need one specialist, or combined work from many specialists?

| Question | Use |
|---|---|
| One department can answer | Handoff |
| Multiple specialists must contribute | Agent-as-Tool |
| User should continue with specialist | Handoff |
| Main Agent should stay in control | Agent-as-Tool |
| Query routing problem | Handoff |
| Report generation problem | Agent-as-Tool |
| Customer support escalation | Handoff |
| Research/teaching pack creation | Agent-as-Tool |


# 15. Enterprise Example

The same concept applies outside education.

## Banking Example: Handoff

```text
Customer
   ↓
Banking Reception Agent
   ├── Loan Agent
   ├── Credit Card Agent
   ├── Fraud Agent
   └── Payment Agent
```

If the customer says:

> "My card is blocked."

The request should go to the Credit Card Agent.

---

## Banking Example: Agent-as-Tool

If a manager asks:

> "Prepare a risk summary for this customer."

The orchestrator may call:

- Loan History Agent
- Transaction Pattern Agent
- Fraud Signal Agent
- Compliance Agent

Then it combines the result into one risk summary.


# 16. Classroom Discussion Exercise

Read the following scenarios and decide the best pattern.

| Scenario | Handoff or Agent-as-Tool? |
|---|---|
| Student asks about hostel fees | ? |
| Faculty asks for lesson plan + quiz + lab exercise | ? |
| Student asks about placement training | ? |
| HOD asks for a course improvement report using feedback, marks, and attendance | ? |
| Customer asks why payment failed | ? |
| Compliance manager asks for complete audit summary | ? |

Discuss for two minutes.


# 17. Hands-on Exercise

Create an **AI Admission Office**.

You need to design:

1. Admission Enquiry Agent
2. Fee Enquiry Agent
3. Hostel Enquiry Agent
4. Course Counselling Agent

Then decide:

- Which queries should use Handoff?
- Which queries should use Agent-as-Tool?

Example query:

> "I want to know course options, fees, hostel availability, and admission steps."

This likely needs Agent-as-Tool because multiple specialists must contribute.


In [ ]:
# Exercise Starter Code

admission_agent = Agent(
    name="Admission Enquiry Agent",
    instructions="""
    TODO: Write instructions for answering admission process questions.
    """,
)

hostel_agent = Agent(
    name="Hostel Enquiry Agent",
    instructions="""
    TODO: Write instructions for answering hostel-related questions.
    """,
)

course_counselling_agent = Agent(
    name="Course Counselling Agent",
    instructions="""
    TODO: Write instructions for suggesting suitable courses.
    """,
)

# TODO:
# 1. Create a simple handoff router for admission queries.
# 2. Create an orchestrator that combines admission + fee + hostel + course counselling responses.


# 18. Challenge Exercise

Extend the teaching orchestrator by adding a fourth Agent:

## Case Study Agent

This Agent should generate a short classroom case study.

Your final teaching pack should include:

1. Concept explanation
2. Quiz
3. Live demo idea
4. Case study

This is a perfect example of Agent-as-Tool.


# 19. Common Mistakes

Avoid these mistakes:

1. Using Handoff when the task needs combined output.
2. Using Agent-as-Tool when the user simply needs one specialist.
3. Creating too many Agents without clear roles.
4. Giving all Agents similar instructions.
5. Not explaining who owns the final response.
6. Forgetting that orchestration logic matters.
7. Making the example too complex for beginners.


# 20. Key Takeaways

In this notebook, we learned:

- **Handoff** routes the user to the right specialist.
- **Agent-as-Tool** uses specialist Agents internally.
- Handoff is like a reception desk.
- Agent-as-Tool is like a project manager.
- Handoff is useful for support, escalation, and routing.
- Agent-as-Tool is useful for reports, teaching packs, research summaries, and multi-step synthesis.
- Good Agent design starts with deciding the collaboration pattern.

---

## Final Mental Model

```text
Handoff:
User → Router → One Specialist → Response

Agent-as-Tool:
User → Orchestrator → Many Specialists → Combined Response
```

Choose the pattern based on the problem.
